In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator/')

from tqdm import tqdm
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProb
from annotator.stqutils import trainClassifier, showMolecularData

import zarr
import tifffile
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

In [ ]:
samples = ['JDC-WP-001-s', 'JDC-WP-001-c', 'JDC-WP-002-r', 'JDC-WP-002-c', 'JDC-WP-003-ae', 'JDC-WP-003-d', 'JDC-WP-004-y', 'JDC-WP-004-c',
       'JDC-WP-005-n', 'JDC-WP-005-c', 'JDC-WP-007-o', 'JDC-WP-007-c', 'JDC-WP-008-v', 'JDC-WP-008-b', 'JDC-WP-009-n', 'JDC-WP-009-c',
       'JDC-WP-010-w', 'JDC-WP-010-c', 'JDC-WP-011-b', 'JDC-WP-011-w', 'JDC-WP-002-v', 'JDC-WP-012-ae', 'JDC-WP-012-c', 'JDC-WP-012-w',
       'JDC-WP-013-y', 'JDC-WP-013-b', 'JDC-WP-014-aj', 'JDC-WP-014-d', 'JDC-WP-015-n', 'JDC-WP-015-b', 'JDC-WP-016-x', 'JDC-WP-016-b']

In [ ]:
outsSTQPath = '/projects/activities/kappsen-tmc/USERS/domans/results-STQ-post-xenium-32-final/'
df_ids = pd.read_csv('/sdata/activities/kappsen-tmc/visium/analysis/downstream/Pancreas Xenium identifiers and paths.csv', index_col=0)
IDsToSTQnames = df_ids['HE-slide-identifier'].to_dict()

ads = {sample: loadAd(outsSTQPath + IDsToSTQnames[sample] + '/', L=None, fname='img.data.ctranspath-1.h5ad', suffix=None)[0] for sample in tqdm(samples)}

In [ ]:
# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=8, spacing=56/0.25, sample_id=sample) for sample in tqdm(samples)], axis=0)

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [ ]:
imagingPath = '/sdata/activities/kappsen-tmc/visium/analysis/downstream/pancreas-Xenium-pre-processed-32-imaging-v2/'

df_pe_tile = pd.concat([pd.read_parquet(imagingPath + f'{id}-df_pe_tile.parquet') for id in tqdm(samples)], keys=samples)
df_ge_tile = pd.concat([pd.read_parquet(imagingPath + f'{id}-df_ge_tile.parquet') for id in tqdm(samples)], keys=samples)
# df_connec_tile = pd.concat([(df_temp := pd.read_parquet(imagingPath + f'{id}-df_connection.parquet').set_index('tile')[['spx']]).loc[~df_temp.index.duplicated()] for id in tqdm(samples)], keys=samples)
df_ctcoarse_tile = pd.concat([pd.read_parquet(imagingPath + f'{id}-df_ctcoarse_tile_is_fraction_False.parquet') for id in samples], keys=samples)
df_ctfine_tile = pd.concat([pd.read_parquet(imagingPath + f'{id}-df_ctfine_tile_is_fraction_False.parquet') for id in samples], keys=samples)
df_ct_tile = pd.concat([df_ctcoarse_tile.drop('Total', axis=1), df_ctfine_tile], axis=1)
df_pe_tile.index.names = ['sample', 'barcode']
df_ge_tile.index.names = ['sample', 'barcode']
df_ct_tile.index.names = ['sample', 'barcode']

In [ ]:
# Add path_id and patch_size to df_pe_tile, df_ge_tile, df_ct_tile indices

df_temp = patchCoordinates.reindex(df_pe_tile.index)[['patch', 'patch_size']]
df_temp['patch'] = df_temp['patch'].fillna('patch_na').astype(str)
df_temp['patch_size'] = df_temp['patch_size'].fillna(0).astype(int)
df_pe_tile['patch'] = df_temp['patch'].values
df_pe_tile['patch_size'] = df_temp['patch_size'].values
df_pe_tile = df_pe_tile.reset_index().set_index(['sample', 'patch', 'barcode']).sort_index()

df_temp = patchCoordinates.reindex(df_ge_tile.index)[['patch', 'patch_size']]
df_temp['patch'] = df_temp['patch'].fillna('patch_na').astype(str)
df_temp['patch_size'] = df_temp['patch_size'].fillna(0).astype(int)
df_ge_tile['patch'] = df_temp['patch'].values
df_ge_tile['patch_size'] = df_temp['patch_size'].values
df_ge_tile = df_ge_tile.reset_index().set_index(['sample', 'patch', 'barcode']).sort_index()

df_temp = patchCoordinates.reindex(df_ct_tile.index)[['patch', 'patch_size']]
df_temp['patch'] = df_temp['patch'].fillna('patch_na').astype(str)
df_temp['patch_size'] = df_temp['patch_size'].fillna(0).astype(int)
df_ct_tile['patch'] = df_temp['patch'].values
df_ct_tile['patch_size'] = df_temp['patch_size'].values
df_ct_tile = df_ct_tile.reset_index().set_index(['sample', 'patch', 'barcode']).sort_index()

In [ ]:
df_ct_gb = df_ct_tile.loc[df_ct_tile['patch_size']>=50].groupby(['sample', 'patch']).sum().drop('patch_size', axis=1)
df_ct_gb = df_ct_gb.loc[(df_ct_gb['Total'] > 1000) & (df_ct_gb['Total'] <= 5000)]

positive_patches = df_ct_gb.index[(df_ct_gb['Endocrine (coarse)']>=75)]
negative_patches = df_ct_gb.index[(df_ct_gb['Endocrine (coarse)']==0)]

print(f'positive_patches: {len(positive_patches)}, negative_patches: {len(negative_patches)}')

# Take at most Nmax positive and Nmax negative patches at random for training
if True:
    Nmax = 1000
    seed = 42
    if len(positive_patches) > Nmax:
        positive_patches = positive_patches[np.sort(np.random.choice(len(positive_patches), Nmax, replace=False))]
    if len(negative_patches) > Nmax:
        negative_patches = negative_patches[np.sort(np.random.choice(len(negative_patches), Nmax, replace=False))]

print(f'positive_patches: {len(positive_patches)}, negative_patches: {len(negative_patches)}')
annotation_results = {p: 'positive' for p in positive_patches.values}
annotation_results.update({p: 'negative' for p in negative_patches.values})

In [ ]:
clfNon = trainClassifier(annotation_results, patchesCDFs)

In [ ]:
x_non, y_non, p_non = inferProb(ads['JDC-WP-012-w'], clfNon, qs, tsize=56/0.25, R=1)
showProb(x_non, y_non, p_non, s=1, marker='s', figsize=(6, 6), colorbar=True, vmin=0., vmax=1, title=r'Regular')

x_non, y_non, p_non = inferProb(ads['JDC-WP-002-v'], clfNon, qs, tsize=56/0.25, R=1)
showProb(x_non, y_non, p_non, s=1, marker='s', figsize=(6, 6), colorbar=True, vmin=0., vmax=1, title=r'Regular')

In [ ]:
alpha = 0.95

clf = trainClassifier(annotation_results, patchesCDFs, alpha=alpha, seed=0, augFunc=PCMA)

In [ ]:
x, y, p = inferProb(ads['JDC-WP-012-w'], clf, qs, tsize=56/0.25, R=1)
showProb(x, y, p, s=1, marker='s', figsize=(6, 6), colorbar=True, vmin=0., vmax=1, title=r'PCMA $\alpha$=' + str(alpha))

x, y, p = inferProb(ads['JDC-WP-002-v'], clf, qs, tsize=56/0.25, R=1)
showProb(x, y, p, s=1, marker='s', figsize=(6, 6), colorbar=True, vmin=0., vmax=1, title=r'PCMA $\alpha$=' + str(alpha))

In [ ]:
showMolecularData_Islets((sample := 'JDC-WP-012-w'), patchCoordinates, df_ct_tile.xs(sample, level='sample')['Endocrine (coarse)'].droplevel('patch'))
showMolecularData_Islets((sample := 'JDC-WP-002-v'), patchCoordinates, df_ct_tile.xs(sample, level='sample')['Endocrine (coarse)'].droplevel('patch'))